In [22]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import SabreSwap, SabreLayout
from qiskit import transpile
from qiskit import qasm3
from qiskit.circuit.controlflow import IfElseOp
from qiskit.transpiler import PassManager, Layout, CouplingMap
from qiskit.transpiler.passes import SetLayout, ApplyLayout, SabreSwap

In [23]:
from qpu.src.load_backend import load_backend_edges
from src.parser import build_structured_trace_from_circuit, format_structured_trace

In [24]:
qc = QuantumCircuit(127, 127)

# Initial gates
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.cx(3, 4)
qc.cx(2, 4)
qc.cx(0, 3)


# Measure qubits needed for the conditional
qc.measure(0, 0)
qc.measure(1, 1)


for_body = QuantumCircuit(127, 127)
for_body.h(4)  # Example operation in the for loop body
for_body.cx(4, 0)
for_body.cx(2, 3)
for_body.cx(1, 4)
for_body.cx(1, 2)

# Add the for loop to the main circuit
qc.for_loop(range(3), None, for_body, qc.qubits, qc.clbits)

# Final measurement
qc.measure(3, 3)

# Create a simple if operation (no else clause)
if_body = QuantumCircuit(127, 127)
if_body.y(4)
if_body.cx(4, 2)
if_body.cx(1, 4)
if_body.cx(1, 25)

# Add the if operation to the circuit
qc.if_test((qc.clbits[2], True), if_body, qc.qubits, qc.clbits)

# Create an if-else operation using IfElseOp
if_else_body = QuantumCircuit(127, 127)
if_else_body.x(2)
if_else_body.cx(2, 1)
if_else_body.cx(0, 2)

else_body = QuantumCircuit(127, 127)
else_body.z(1)
else_body.cx(1, 0)
else_body.cx(3, 1)

# Add the if-else operation to the circuit using IfElseOp
if_else_op = IfElseOp((qc.clbits[1], True),
                      if_else_body, else_body)
qc.append(if_else_op, qc.qubits, qc.clbits)

# Create a for loop body that contains an if-else operation
for_body_with_if_else = QuantumCircuit(127, 127)

# Create the if body inside the for loop
if_body_in_for = QuantumCircuit(127, 127)
if_body_in_for.h(3)
if_body_in_for.cx(3, 18)
if_body_in_for.cx(1, 10)

# Create the else body inside the for loop
else_body_in_for = QuantumCircuit(127, 127)
else_body_in_for.z(1)
else_body_in_for.cx(1, 7)
else_body_in_for.cx(3, 20)

# Add the if-else operation inside the for loop body
if_else_op_in_for = IfElseOp((qc.clbits[0], True),
                             if_body_in_for, else_body_in_for)
for_body_with_if_else.append(
    if_else_op_in_for, for_body_with_if_else.qubits, for_body_with_if_else.clbits)

# Add the for loop with if-else inside to the main circuit
qc.for_loop(range(2), None, for_body_with_if_else, qc.qubits, qc.clbits)

# Final measurement
qc.measure(3, 3)

qc.cx(0, 1)
qc.cx(1, 2)

In [40]:
qc = qasm3.load("../d-queko/benchmarks/16qbt/queko-016qbt_nest_01_nodes010_leaf-depth-10/circ_00.qasm")

In [41]:
from qiskit.transpiler import CouplingMap

# Get the backend's coupling map from loaded edges
edges = load_backend_edges(backend_name="ibm_sherbrooke")
coupling_map = CouplingMap(edges)

In [17]:

# qc = qasm3.load(r"C:\Users\ASUS ROG\OneDrive\Desktop\dev\Quantum\PFE\quantum-compiler\d-queko\benchmarks\16qbt\queko-016qbt_nest_01_nodes010_leaf-depth-40\circ_00.qasm")

# Transpile the circuit using Sabre with trivial layout and no optimization
transpiled_qc = transpile(
    qc,
    coupling_map=coupling_map,
    optimization_level=0,
    layout_method='trivial',
    routing_method='sabre'
)



In [42]:
layout = Layout({qb: i for i, qb in enumerate(qc.qubits)})

pm = PassManager([
    SetLayout(layout),                  # set (but don't apply) the layout
    SabreSwap(coupling_map=coupling_map,
              heuristic='decay'),  # insert SWAPs only
    ApplyLayout(),                      # relabel to physical qubits at the end
])

routed_qc = pm.run(qc)

In [43]:
# Export to QASM3 string
qasm3_string = qasm3.dumps(routed_qc)
print(qasm3_string)

with open("routed_circuit.qasm", "w") as f:
    f.write(qasm3_string)


OPENQASM 3.0;
include "stdgates.inc";
bit[4] c;
cx $0, $1;
h $0;
rz(1.96349541) $0;
h $1;
rz(1.96349541) $1;
cz $0, $1;
rz(2.74889357) $0;
rx(1.17809725) $0;
x $1;
rz(1.96349541) $1;
cx $0, $1;
cz $0, $1;
cx $0, $1;
x $0;
h $0;
rx(2.74889357) $0;
h $0;
h $0;
rz(0.39269908) $0;
h $1;
z $1;
rx(1.17809725) $1;
z $1;
rz(2.74889357) $1;
z $1;
cz $0, $1;
cx $0, $1;
x $0;
x $0;
rz(2.74889357) $1;
rz(1.17809725) $1;
cz $0, $1;
z $4;
rx(0.39269908) $5;
z $5;
z $5;
rx(0.39269908) $5;
h $5;
rx(0.39269908) $5;
z $6;
rx(0.39269908) $6;
x $6;
rx(2.74889357) $6;
rx(2.74889357) $6;
h $6;
h $6;
rx(2.74889357) $6;
rz(0.39269908) $6;
z $6;
z $6;
h $6;
x $6;
x $6;
rz(0.39269908) $6;
rx(2.74889357) $6;
h $6;
x $6;
rz(1.96349541) $6;
rz(1.96349541) $6;
swap $7, $6;
swap $5, $6;
swap $4, $5;
swap $3, $4;
cz $2, $3;
x $2;
x $3;
cz $2, $3;
cz $2, $3;
cz $2, $3;
rx(0.39269908) $2;
z $2;
x $2;
z $2;
h $2;
z $3;
rx(1.96349541) $3;
x $3;
x $3;
z $3;
cx $2, $3;
cx $2, $3;
cz $2, $3;
h $2;
h $2;
rz(0.39269908) $2;
r

In [44]:
trace = build_structured_trace_from_circuit(
    routed_qc, decompose=False)

In [47]:
# nb swaps in routed_qc
nb_swaps = sum(1 for instr, qargs, cargs in routed_qc.data if instr.name == 'swap')
print(f"Number of SWAPs in routed circuit: {nb_swaps}")

Number of SWAPs in routed circuit: 23


C:\Users\ASUS ROG\AppData\Local\Temp\ipykernel_17352\2071288317.py:2: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 2.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  nb_swaps = sum(1 for instr, qargs, cargs in routed_qc.data if instr.name == 'swap')


In [45]:
print(format_structured_trace(trace))

gate 1 (cx, q0, q1)
gate 2 (h, q0)
gate 3 (rz, q0)
gate 4 (h, q1)
gate 5 (rz, q1)
gate 6 (cz, q0, q1)
gate 7 (rz, q0)
gate 8 (rx, q0)
gate 9 (x, q1)
gate 10 (rz, q1)
gate 11 (cx, q0, q1)
gate 12 (cz, q0, q1)
gate 13 (cx, q0, q1)
gate 14 (x, q0)
gate 15 (h, q0)
gate 16 (rx, q0)
gate 17 (h, q0)
gate 18 (h, q0)
gate 19 (rz, q0)
gate 20 (h, q1)
gate 21 (z, q1)
gate 22 (rx, q1)
gate 23 (z, q1)
gate 24 (rz, q1)
gate 25 (z, q1)
gate 26 (cz, q0, q1)
gate 27 (cx, q0, q1)
gate 28 (x, q0)
gate 29 (x, q0)
gate 30 (rz, q1)
gate 31 (rz, q1)
gate 32 (cz, q0, q1)
gate 33 (z, q4)
gate 34 (rx, q5)
gate 35 (z, q5)
gate 36 (z, q5)
gate 37 (rx, q5)
gate 38 (h, q5)
gate 39 (rx, q5)
gate 40 (z, q6)
gate 41 (rx, q6)
gate 42 (x, q6)
gate 43 (rx, q6)
gate 44 (rx, q6)
gate 45 (h, q6)
gate 46 (h, q6)
gate 47 (rx, q6)
gate 48 (rz, q6)
gate 49 (z, q6)
gate 50 (z, q6)
gate 51 (h, q6)
gate 52 (x, q6)
gate 53 (x, q6)
gate 54 (rz, q6)
gate 55 (rx, q6)
gate 56 (h, q6)
gate 57 (x, q6)
gate 58 (rz, q6)
gate 59 (rz, q6)
ga

In [46]:
with open("structured_traceXX.txt", "w") as f:
    f.write(format_structured_trace(trace))

In [31]:
from src.routing import Qlosure


from qpu.src.load_backend import load_backend_edges
from src.backend import QuantumBackend

In [34]:
from src.dag import build_dag, extract_multi_qubit_dag
dag = build_dag(qc)
dag2q = extract_multi_qubit_dag(dag)

In [35]:
backend = QuantumBackend(edges)
poly_mapper = Qlosure(backend)

In [36]:
qlosure_results = poly_mapper.run(
    dag, dag2q, initial_mapping="trivial", num_iter=1, verbose=True)

Running Qlosure:  75%|███████▌  | 12/16 [00:00<00:00, 62.57it/s]


In [37]:
trace = poly_mapper.get_structured_trace()

In [39]:
print(poly_mapper.format_structured_trace(trace))

gate 87 (h, q0)
gate 88 (cx, q3, q4)
gate 89 (cx, q0, q1)
gate 90 (cx, q1, q2)
gate 91 (measure, q1)
gate 92 (swap, q3, q2)
gate 93 (cx, q2, q4)
gate 94 (swap, q1, q3)
gate 95 (cx, q0, q3)
gate 96 (measure, q0)
gate 97 (swap, q1, q2)
gate 98 (swap, q3, q1)
gate 99 (swap, q0, q3)
gate 100 (swap, q1, q2)
for (iterations=100) {
  gate 101 (h, q4)
  gate 102 (cx, q2, q3)
  gate 103 (cx, q4, q0)
  gate 104 (swap, q0, q4)
  gate 105 (cx, q1, q4)
  gate 106 (cx, q1, q2)
  gate 107 (swap, q4, q0)
}
gate 108 (measure, q3)
if {
  // then
  gate 109 (y, q4)
  gate 110 (cx, q4, q2)
  gate 111 (swap, q2, q1)
  gate 112 (cx, q1, q4)
  gate 113 (swap, q4, q1)
  gate 114 (swap, q24, q25)
  gate 115 (swap, q15, q1)
  gate 116 (swap, q22, q1)
  gate 117 (swap, q23, q1)
  gate 118 (cx, q1, q25)
  gate 119 (swap, q23, q1)
  gate 120 (swap, q25, q24)
  gate 121 (swap, q22, q23)
  gate 122 (swap, q15, q22)
  gate 123 (swap, q4, q15)
  gate 124 (swap, q2, q4)
}
if {
  // then
  gate 125 (x, q2)
  gate 126 (c